In [ ]:
from pyscf import gto, scf, cc
from qarp.operators import JordanWigner
from qarp.operators.pyscf import fermion_operator_from_mf, onv_from_mf

from qarp.blocks import ComputationalBasisStateBlock, UCCBlock, CompositeBlock
from qarp.algorithms import StateVector

mol = gto.M(atom="H 0 0 0; H 0 0 1; H 0 0 11; H 0 0 12", basis="sto3g")
mol.build()
mf = scf.RHF(mol)
mf.kernel()
mycc = cc.CCSD(mf)
mycc.kernel()

fop = fermion_operator_from_mf(mf)
onv = onv_from_mf(mf)

qham = JordanWigner().encode_operator(fop)

ref = ComputationalBasisStateBlock(onv)
ucc = UCCBlock(onv, singles=True, doubles=True)
ansatz = CompositeBlock([ref, ucc])
ansatz.build()
ansatz.plot()


In [ ]:
from qarp.engines import QarpEngine
param = dict(zip(ansatz.symbols, [0.] * len(ansatz.symbols)))

sv = StateVector(bra=ansatz, operator=qham, ket=ansatz)
engine = QarpEngine()
engine.build([sv])

print(engine.run(param))
print(engine.run_gradient(param))

In [ ]:
from qarp.algorithms import VQE

vqe = VQE(operator=qham, ket=ansatz, verbose=True, gradient=True, initial_parameters=[0]*len(ansatz.symbols))
vqe.build()
e, p = vqe.run()

In [ ]:
print(mycc.e_tot)